In [ ]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import matplotlib.pyplot as plt

pio_fields= ["participants", "interventions", "outcomes"]


In [ ]:
#making gold csv - 
import os
import pandas as pd


# extract spans from 0/1 labels
def get_spans(tokens, labels):
    spans = []
    current = []

    for tok, lab in zip(tokens, labels):
        lab = int(lab)

        if lab != 0:
            current.append(tok)
        else:
            if current:
                spans.append(" ".join(current))
                current = []

    if current:
        spans.append(" ".join(current))

    return spans


# load token file
def read_tokens(doc_id, base_path):
    path = os.path.join(base_path, "documents", f"{doc_id}.tokens")
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]


# load label file
def read_labels(doc_id, base_path, field, split="test"):
    if split == "test":
        split = "test/gold"

    path = os.path.join(
        base_path,
        "annotations",
        "aggregated",
        "starting_spans",
        field,
        split,
        f"{doc_id}.AGGREGATED.ann"
    )

    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]


# build one row
def build_one_doc(doc_id, base_path):
    tokens = read_tokens(doc_id, base_path)

    row = {"doc_id": doc_id}

    for field in ["participants", "interventions", "outcomes"]:
        labels = read_labels(doc_id, base_path, field)
        spans = get_spans(tokens, labels)

        # join spans into one string
        row[f"{field}_gold"] = " ; ".join(spans)

    return row


# build full gold dataframe
def build_gold(doc_ids, base_path):
    rows = []

    for d in doc_ids:
        try:
            rows.append(build_one_doc(d, base_path))
        except:
            continue  # skip broken files

    return pd.DataFrame(rows)


# run once -
# DATA_PATH = "./ebm_nlp_2_00"
# gold_df = build_gold(test_doc_ids_p, DATA_PATH)
# gold_df.to_csv("gold_structured.csv", index=False)

In [ ]:
''' EDIT AFTER I GET CSV FILE-

def scrub_text(txt):
    # light cleaning before comparison
    if pd.isna(txt):
        return ""
    
    txt = str(txt).lower().strip()
    txt = re.sub(r"[^\w\s]", " ", txt)
    txt = re.sub(r"\s+", " ", txt)
    
    return txt


def load_single_variant_csv(csv_path, variant_label):
    # load one csv for one method
    tab = pd.read_csv(csv_path)
    tab["variant"] = variant_label
    return tab


def join_gold_and_preds(gold_csv_path, pred_csv_path, variant_label):
    # join separate gold and prediction files
    gold_tab = pd.read_csv(gold_csv_path)
    pred_tab = pd.read_csv(pred_csv_path)
    
    merged = gold_tab.merge(pred_tab, on="doc_id", how="inner")
    merged["variant"] = variant_label
    
    return merged


def stack_variants(list_of_tabs):
    # combine multiple methods into one dataframe
    return pd.concat(list_of_tabs, ignore_index=True)


def check_expected_columns(tab, fields):
    # quick check for expected columns
    needed = ["doc_id", "variant"]
    
    for fld in fields:
        needed.append(f"{fld}_gold")
        needed.append(f"{fld}_pred")
    
    missing = [c for c in needed if c not in tab.columns]
    
    if missing:
        print("Missing columns:")
        for c in missing:
            print("-", c)
    else:
        print("All expected columns are present.")

        
#examples
tab_bert = join_gold_and_preds(
    "gold_structured.csv",
    "bert_predictions.csv",
    "bert"
)

tab_rule = join_gold_and_preds(
    "gold_structured.csv",
    "rule_predictions.csv",
    "rule"
)

all_runs = stack_variants([tab_bert, tab_rule])

check_expected_columns(all_runs, pio_fields)

'''

In [ ]:
##evaluation

#exact match
def exact_m(gold_txt,pred_txt):
    g=scrub_text(gold_txt)
    p=scrub_text(pred_txt)
    return int(g != "" and g==p)

#1.token overlap F1
def overlap_f1(gold_txt, pred_txt):
#match on overlapping token
g=scrub_text(gold_txt)
p=scrub_text(pred_txt)

if g== "" and p== "":
    return 1.0
if g== "" or p== "":
    return 0.0

gold_tkn=g.split()
pred_tkn=p.split()

gold_count=defaultdict(int)
pred_count=defaultdict(int)

for w in gold_tkn:
    gold_count[w] +=1
for w in pred_tkn:
    pred_count[w] +=1

shared=0
for w in gold_count:
    shared =shared + min(gold_count[w],pred_count[w])

if shared==0:
    return 0.0

prec=shared/len(pred_tkn)
rec=shared/len(gold_tkn)

return 2*prec*rec/(prec+rec)

In [ ]:
#evaluate one field
def evaluate_1field(tab,gold_col,pred_col,match_threshold=0.5):

    #calculate metrics 
    gold_nonempty=0
    pred_nonempty=0
    covered=0
    tp=0
    fp=0
    fn=0
    exact_score=[]
    s_score=[]

    for _, row in tab.iterrow():
        g=scrub_text(row[gold_col])
        p=scrub_text(row[pred_col])
        exact_score.append(exact_m(g,p))
        #partial match
        s_score.append(overlap_f1(g,p))


        #checks if predicted the answer and if its true
        has_gold=g!= ""
        has_pred=p!= ""
        if has_gold:
            gold_nonempty+=1
        if has_pred:
            pred_nonempty+=1
        if has_gold and has_pred:
            covered+=1

        #check if prediction is close enough to overlap_f1
        good=overlap_f1(g,p)>=match_threshold
        if has_gold and has_pred and good:
            tp+=1
        elif has_pred and (not has_gold or not good):
            fp+=1
        if has_gold and (not has_pred or not good):
            fn+=1

        precision=tp/(tp+fp) if (tp+fp)>0 else 0.0
        recall=tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1=2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
        coverage=covered/gold_nonempty if gold_nonempty>0 else 0.0

        return{"field":gold_col.replace("_gold",""),"exact_match":np.mean(exact_score),
               "avg_overlap_f1":np.mean(s_score),"precision":precision,
               "recall":recall, "f1":f1, "coverage":coverage}
    
    

        

In [ ]:
#evaluate one variant
def eval_variant(tab,fields,variant_name=None, match_threshold=0.5):
    #evaluate across all fields
    rows=[]
    if variant_name is None:
        variant_name=tab["variant"].iloc[0] if "variant" in tab.columns else "unknown"
    for fld in fields:
        row=evaluate_1field(tab, gold_col=f"{fld}_gold", pred_col=f"{fld}_pred",
                            match_threshold=match_threshold)
        row["variant"]=variant_name
        rows.append(row)

    output=pd.DataFrame(rows)
    overall={"variant":variant_name, "field":"overall_avg","exact_match":output["exact_match"].mean(),
           "avg_f1_overlap":output["avg_overlap_f1"].mean(),"precision":output["precision"].mean(),
           "recall":output["recall"].mean(),"f1":output["f1"].mean(),"coverage":output["coverage"].mean()}
    return pd.concat([output,pd.DataFrame([overall])], ignore_index=True)


#evaluating all variants
def evaluate_all_variants(all_tab, fields,match_thresholds=0.5):
    results=[]
    for name,sub in all_tab.groupby("variant"):
        res=eval_variant(sub, fields,name,match_thresholds)
        results.append(res)
    return pd.concat(results,ignore_index=True)

#plot coverage v precision
def plot_cvp(score_tab):
    show=score_tab[score_tab["field"] !="overall_avg"]
    plt.figure(figsize=(7,5))
    for name,grp in show.groupby("variant"):
        plt.scatter(grp["coverage"],grp["precision"], label=name)
    
    plt.xlabel("Coverage")
    plt.ylabel("Precision")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
##3.downstream usability

#making table
def make_q_table(tab,use_pred=False):
    suffix="pred" if use_pred else "_gold"
    small_tab=tab[["doc_id", f"participants{suffix}", f"interventions{suffix}",
                   f"outcomes{suffix}"]].copy()
    small_tab.columns=["doc_id","participants","interventions","outcomes"]
    return small_tab

#scoring one doc for one query
def score_1doc(row,must_have):
    score=0
    for field,words in must_have.items():
        text=scrub_text(row[field])

        for w in words:
            w2=scrub_text(w)
            if re.search(rf"\b{re.escape(w2)}\b",text):
                score+=1
    return score

#rank doc for one query
def rank_doc(tab,must_have):
    work=tab.copy()
    work["score"]=work.apply(lambda row:score_1doc(row,must_have),axis=1)
    work=work[work["score"]>0].copy()
    work=work.sort_values(["score","doc_id"],ascending=[False, True])
    return work[["doc_id","score"]]

#one ranked query for one method
def eval_1rq(gold_tab,pred_tab,query_item,top_k=10):
    gold_rank=rank_doc(gold_tab,query_item["must_have"])
    pred_rank=rank_doc(pred_tab,query_item["must_have"])

    gold_top=set(gold_rank.head(top_k)["doc_id"].tolist())
    pred_top=set(pred_rank.head(top_k)["doc_id"].tolist())
    tp=len(gold_top&pred_top)
    fp=len(pred_top-gold_top)
    fn=len(gold_top-pred_top)

    precision=tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall=tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1=2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
    return{"query":query_item["name"],"precision":precision,"recall":recall,"f1":f1,
           "gold_matches":len(gold_top),"pred_matches":len(pred_top)}






In [ ]:
#all queries for one method
def eval_qranked(gold_tab,pred_tab,queries,top_k=10):
    rows=[]
    for q in queries:
        row=eval_1rq(gold_tab,pred_tab,q,top_k=top_k)
        rows.append(row)
    output=pd.DataFrame(rows)

    overall=pd.DataFrame([{"query":"overall","precision":output["precision"].mean(),
                           "recall":output["recall"].mean(),"f1":output["f1"].mean(),
                           "gold_match":output["gold_matches"],"pred_match":output["pred_matches"].sum()}])
    return pd.concat([output,overall],ignore_index=True)

#evaluate all method
def eval_q_allvariant(all_tab,queries,top_k=10):
    all_rows=[]
    for v_name,sub_tab in all_tab.groupby("variant"):
        gold_q=make_q_table(sub_tab, use_pred=False)
        pred_q=make_q_table(sub_tab,use_pred=True)
        res=eval_qranked(gold_q,pred_q,queries,top_k=top_k)
        res["variant"]=v_name
        all_rows.append(res)
    return pd.concat(all_rows,ignore_index=True)



In [ ]:
#queries
#example query

queries=[
    {"name":"diabetes participants","must_have":{"participants":["diabetes"]}},
    {"name":"exercise interventions","must_have":{"interventions":["exercise"]}},
    {"name":"blood pressure outcomes","must_have":{"outcome":["blood pressure"]}},
    {"name":"diabetes and metaformin","must_have":{"participants":["diabetes"], "interventions":["metaformin"]}},
    {"name":"exercise and blood pressure","must_have":{"interventions":["exercise"],"outcomes":["blood pressure"]}}
    
]

#all run defined when i get csv file
query_score_all=eval_q_allvariant(all_run,queries,top_k_10)
print(query_score_all)